# Keyword spam model MVP

## Setup

You'll need to download en_core_web_sm for this task

In [1]:
# ! uv pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

In [2]:
# import nltk

# nltk.download("punkt_tab")

In [3]:
from itertools import chain
import os
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy
import truecase
import xgboost as xgb
import unidecode

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
nlp = spacy.load("en_core_web_sm")

## Useful feature engineering functions

In [5]:
def proportion_of_stopwords(description):
    tokens = description.split(" ")
    num_stopwords = len(
        [word for word in tokens if word.lower() in nlp.Defaults.stop_words]
    )
    return float(num_stopwords) / float(len(tokens)) if len(tokens) else 0.0


def average_length_of_word(description):
    tokens = description.split(" ")
    return np.mean([len(word) for word in tokens]) if len(tokens) else 0.0


# Only if the entire word is digit will this consider the word being digit
# Ignore words where digit is part of the word e.g. y2k [Done]
def proportion_of_numbers(description):
    tokens = description.split(" ")
    num_digits = len([word for word in tokens if word.isdigit()])
    return float(num_digits) / float(len(tokens)) if len(tokens) else 0.0


def normalise_nonascii_chars(input_str):
    return unidecode.unidecode(input_str)


# This is removing hash tags. Spammy descriptions could have lots of hashtags
# Could be removing legitimate signals [Done]
def replace_special_chars(main_string):
    return re.sub("[,;@#!\?\+\*\n\-: /]", " ", main_string)


def keep_alphanumeric_chars(string_input):
    return re.sub("[^A-Za-z0-9& ]", "", string_input)


def remove_spaces(string_input):
    return " ".join(string_input.split())


def lemmatize(string_input):
    token_object = nlp(string_input)
    lemmas_list = [
        word.lemma_ if word.lemma_ != "-PRON-" else word.text for word in token_object
    ]
    return " ".join(lemmas_list)


def drop_digits(s):
    return "".join([i for i in s if not i.isdigit()])


def clean_description(input_str):
    input_str = replace_special_chars(input_str.lower())
    input_str = normalise_nonascii_chars(input_str)
    input_str = keep_alphanumeric_chars(input_str)
    input_str = lemmatize(input_str)
    input_str = remove_spaces(input_str)
    return input_str

## SK Learn Pipeline

In [9]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer


class FeatureEngineering(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        df = X.copy()
        # Clean description
        df["description"] = df["description"].apply(clean_description)
        # Feature engineering on cleaned decription
        df["proportion_of_stopwords"] = df["description"].apply(proportion_of_stopwords)
        df["average_length_of_word"] = df["description"].apply(average_length_of_word)
        df["proportion_of_numbers"] = df["description"].apply(proportion_of_numbers)
        # Drop digits, true case, then get perform entity recognition
        df["description"] = df["description"].apply(drop_digits)
        df["description_truecase"] = df["description"].apply(truecase.get_true_case)
        df["description_nlp"] = df["description_truecase"].apply(nlp)

        df["named_entities"] = ""

        for i, description_nlp in df["description_nlp"].items():
            named_entities_sets = description_nlp.ents
            named_entities = list(set(chain(*named_entities_sets)))
            df["named_entities"].at[i] = " ".join(j.text for j in named_entities)

        return df


column_transformer = ColumnTransformer(
    transformers=[
        (
            "named_entities_tfidf",
            TfidfVectorizer(
                stop_words=list(nlp.Defaults.stop_words),
                max_features=500,
            ),
            "named_entities",
        ),
        (
            "description_tfidf",
            TfidfVectorizer(
                stop_words=list(nlp.Defaults.stop_words),
                max_features=500,
            ),
            "description",
        ),
        (
            "manual_features",
            "passthrough",
            [
                "product_id",
                "proportion_of_stopwords",
                "average_length_of_word",
                "proportion_of_numbers",
            ],
        ),
    ],
    remainder="drop",
)


class DenseTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        return X.toarray() if hasattr(X, "toarray") else X

In [10]:
from sklearn.pipeline import Pipeline

feature_processing_pipeline = Pipeline(
    steps=[
        ("feature_engineering", FeatureEngineering()),
        ("features", column_transformer),
        ("to_dense", DenseTransformer()),
    ]
)

In [11]:
cwd = os.getcwd()

df_train = pd.read_csv(os.path.join(cwd, "../data/train_set.tsv"), sep="\t")[
    ["product_id", "description", "label"]
]

df_train = df_train.reset_index(drop=True)

X_train = df_train.drop(columns=["label"]).copy()
y_train = df_train["label"].copy()

print("Transforming X_train..")
X_train = feature_processing_pipeline.fit_transform(X_train)

print("Reading test data..")
# Do not touch df_test until final eval to avoid data leakage
df_test = pd.read_csv(os.path.join(cwd, "../data/test_set.tsv"), sep="\t")[
    ["product_id", "description", "label"]
]
X_test = df_test.drop(columns=["label"]).copy()
y_test = df_test["label"].copy()

print("Transform X_test without fitting...")
X_test = feature_processing_pipeline.transform(X_test)

Transforming X_train..


/Users/shengy/Documents/GitHub/depop_takehome_task/.venv/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


Reading test data..
Transform X_test without fitting...


## Fit model

In [12]:
# No hyper parameter tuning? [Done]
# Also - eval_set here should be carved from training set, leaving eval set purely for unseen eval to avoid leakage
# Also n_estimators = 2 -> not enough [Done]
param_dist = {
    "objective": "binary:logistic",
    "n_estimators": 2,
    "eval_metric": "logloss",
}

clf = xgb.XGBClassifier(**param_dist)

clf.fit(
    X_train,
    y_train,
    # eval_set=[(X_train, y_train)], # We need to carve out eval set from X_train and X_test
    verbose=True,
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=2, n_jobs=None,
              num_parallel_tree=None, ...)

## Predict on model

In [13]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

## Evaluate

In [14]:
# Data set mean is 58%? Is this a realisitic expectation for distribution of spams? I'd expect spams to be less prevalent
# If training set distribution is not the same as real world distribution -> calibration of model will be off..
print(df_train["label"].mean())
print(df_test["label"].mean())

0.598
0.528


In [15]:
# Eval on accuracy is meaningless here due to data leakage
# Also is accuracy the right metric here for two reasons
# Precision -> as false positive would cost legitimate seller revenue -> bad experience for sellers
# At the least report F1 score (if undecided between precision/ recall) [Done]

# Plus we're working with ranking problem here? Do we want to treat a 51% vs 90% prediction the same way?
# Or do we want to do risk-based actions, e.g. 60%++ automated removal, 40-60% for human in the loop eval, 40% not spam?
# If so, then is this a ranking problem -> we should report on log loss + checking if probabilities are calibrated [Done]


from sklearn.metrics import accuracy_score

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Train Accuracy: {(train_accuracy * 100.0):.2f}%")
print(f"Test Accuracy: {(test_accuracy * 100.0):.2f}%")

Train Accuracy: 90.27%
Test Accuracy: 87.20%


## Oh my goodness, I'm a brilliant Data Scientist

## Check features

In [16]:
feature_names = feature_processing_pipeline.named_steps[
    "features"
].get_feature_names_out()

df_X_train = pd.DataFrame(
    X_train, columns=feature_names, index=df_train.drop(columns=["label"]).index
)
df_X_test = pd.DataFrame(
    X_test, columns=feature_names, index=df_test.drop(columns=["label"]).index
)

features = pd.DataFrame(
    zip(df_X_train.columns, clf.feature_importances_),
    columns=["feature_name", "feature_importance"],
)
features.sort_values("feature_importance", ascending=False).head(20)

,feature_name,feature_importance
1002,manual_features__average_length_of_word,0.258555
1003,manual_features__proportion_of_numbers,0.103514
234,named_entities_tfidf__jean,0.063425
596,description_tfidf__condition,0.047410
687,description_tfidf__goth,0.046689
806,description_tfidf__nike,0.041026
260,named_entities_tfidf__lauren,0.040974
995,description_tfidf__yk,0.040115
554,description_tfidf__brand,0.038852
862,description_tfidf__question,0.038385


# Summary of identified issues

### 1. Data leakage: [Done]
- train/ test was concatenated, and feature engineering done on all
- then, model was train on all data, and evaluated on test data that was seen during training
- this needs to be fixed, or any evaluation is non realistic
- using sklearn pipelines could make this easier and repeatable to get this into prod: one single pipeline to handle train and test data for inference

### 2. Training issues:
- n_estimator: 2
- product_id used as features
- non-pythonic way of code: using eval()
- complex/ non-repeatable way of feature engineering -> hard to convert into production code -> use Sklearn pipelines at the very least
- no hyperparam tuning

### 3. Evaluation criteria:
- Accuracy isn't the best metric
- We should care about cost asymmetry for different parties:
    - false positive: bad experience for sellers
    - false negative: bad experience for buyers
    - we should compare F1 score at the minimum, or report on both precision and recall to understand trade off
- Also this could be a ranking problem as we may be in taking risk-based approach to handling spams
    - inspect to make sure probabilities are calibrated, and if not, fix calibration prior to reporting on other metrics

### 4. Feature Engineering issues:
- Stripping out a lot of potentially useful signals of spammy texts, e.g. hashtags, digits, only considering unigrams/ ignoring compound terms
- TFIDF resulting in very sparse matrix
- Only very basic feature engineering. We could do a lot more here:
    - Number of spammy terms captured e.g.
    - Number of brands mentioned in description
    - Number of sizing terms mentioned in description
    - Number of styles mentioned in description
    - Number of terms matching previously known spammy terms
    - Total length of description
    - For something to be considered spammy, we want to know how relevant the words in the description is in relevance to the item
    - So we probably want something along those lines -> does this word relate to the item at all?
- Using SpaCY for NER -> is this good for fashion brand though?
    - Do we need explicit brand dictionary or even a tuned brand specific NER to parse out brand names from descriptions?
- Lower casing, then true casing again -> redundant steps
- `proportion_of_numbers`: doesn't work on terms where digits is in the word e.g. y2k
- removing digits remove legitimate terms e.g. y2k -> could there be a better way at handling this